# Document Packet Splitting Evaluation

This notebook demonstrates both metric sets for evaluating document packet splitting:

1. **Classical Metrics** (`DocSplitClassificationMetrics`) — binary exact-match
2. **Proposed Metrics** (`evaluate_packet`) — continuous clustering + ordering scores

Reference: [DocSplit paper (arXiv:2602.15958)](https://arxiv.org/abs/2602.15958)

## Setup

Make sure stickler is installed: `pip install -e .` from the repo root.

In [ ]:
from stickler.doc_split import DocSplitClassificationMetrics, evaluate_packet

## Scenario: 10-Page Lending Packet

A lending packet with 3 documents:
- **1099 form** (pages 0-2)
- **W-2** (pages 3-6)
- **Pay stub** (pages 7-9)

---
## Part 1: Classical Metrics

Section-level input — each dict represents one document in the packet.

In [ ]:
# Ground truth sections
gt_sections = [
    {"section_id": "1099", "document_class": "1099", "page_indices": [0, 1, 2]},
    {"section_id": "w2", "document_class": "w2", "page_indices": [3, 4, 5, 6]},
    {"section_id": "paystub", "document_class": "pay_stub", "page_indices": [7, 8, 9]},
]

### Scenario A: Perfect prediction

In [ ]:
pred_perfect = [
    {"section_id": "p1", "document_class": "1099", "page_indices": [0, 1, 2]},
    {"section_id": "p2", "document_class": "w2", "page_indices": [3, 4, 5, 6]},
    {"section_id": "p3", "document_class": "pay_stub", "page_indices": [7, 8, 9]},
]

m = DocSplitClassificationMetrics()
m.load_sections(gt_sections, pred_perfect)
results = m.calculate_all_metrics()

print(f"Page Level Accuracy:          {results['page_level_accuracy']['accuracy']:.2%}")
print(f"Split Accuracy (no order):    {results['split_accuracy_without_order']['accuracy']:.2%}")
print(f"Split Accuracy (with order):  {results['split_accuracy_with_order']['accuracy']:.2%}")

### Scenario B: Boundary error (page 6 assigned to wrong section)

In [ ]:
pred_boundary_error = [
    {"section_id": "p1", "document_class": "1099", "page_indices": [0, 1, 2]},
    {"section_id": "p2", "document_class": "w2", "page_indices": [3, 4, 5]},
    {"section_id": "p3", "document_class": "pay_stub", "page_indices": [6, 7, 8, 9]},
]

m2 = DocSplitClassificationMetrics()
m2.load_sections(gt_sections, pred_boundary_error)
results2 = m2.calculate_all_metrics()

print(f"Page Level Accuracy:          {results2['page_level_accuracy']['accuracy']:.2%}")
print(f"Split Accuracy (no order):    {results2['split_accuracy_without_order']['accuracy']:.2%}")
print(f"Split Accuracy (with order):  {results2['split_accuracy_with_order']['accuracy']:.2%}")
print()
print("Notice: one boundary error drops split accuracy to 33% (only 1099 matches).")
print("This is a known limitation of classical metrics — no partial credit.")

### Markdown Report

In [ ]:
from IPython.display import Markdown

Markdown(m2.generate_markdown_report(results2))

---
## Part 2: Proposed Metrics (Packet Score)

Page-level input — each dict represents one page in the packet.

### Edge Case: Perfect Match

In [ ]:
perfect_pages = [
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 1, "page_number_predicted": 1, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 2, "page_number_predicted": 2, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 3, "page_number_predicted": 3, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "form-01", "group_id_predicted": "form-01", "page_number": 4, "page_number_predicted": 4, "class_label": "form", "class_label_predicted": "form"},
    {"group_id": "form-01", "group_id_predicted": "form-01", "page_number": 5, "page_number_predicted": 5, "class_label": "form", "class_label_predicted": "form"},
]

r = evaluate_packet(perfect_pages, strict_clustering=True)
print(f"Packet Score:    {r['final_score']:.4f}")
print(f"Clustering:      {r['clustering_score']:.4f}")
print(f"V-measure:       {r['v_measure']:.4f}")
print(f"Rand Index:      {r['rand_index']:.4f}")
print(f"Ordering:        {r['avg_ordering_score']:.4f}")

### Edge Case: Misclassification Only

Correct grouping and ordering, but class labels are swapped.
Classical metrics would report 0% — proposed metrics give partial credit.

In [ ]:
misclass_pages = [
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 1, "page_number_predicted": 1, "class_label": "invoice", "class_label_predicted": "form"},
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 2, "page_number_predicted": 2, "class_label": "invoice", "class_label_predicted": "form"},
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 3, "page_number_predicted": 3, "class_label": "invoice", "class_label_predicted": "form"},
    {"group_id": "form-01", "group_id_predicted": "form-01", "page_number": 4, "page_number_predicted": 4, "class_label": "form", "class_label_predicted": "invoice"},
    {"group_id": "form-01", "group_id_predicted": "form-01", "page_number": 5, "page_number_predicted": 5, "class_label": "form", "class_label_predicted": "invoice"},
]

r = evaluate_packet(misclass_pages, strict_clustering=True)
print(f"Packet Score:    {r['final_score']:.4f}  (classical would be 0%)")
print(f"Clustering:      {r['clustering_score']:.4f}")
print(f"Ordering:        {r['avg_ordering_score']:.4f}  (still perfect — order is correct)")

### Edge Case: Wrong Ordering Only

Correct grouping and classification, but pages scrambled within groups.

In [ ]:
wrong_order_pages = [
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 1, "page_number_predicted": 3, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 2, "page_number_predicted": 1, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 3, "page_number_predicted": 2, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "form-01", "group_id_predicted": "form-01", "page_number": 4, "page_number_predicted": 5, "class_label": "form", "class_label_predicted": "form"},
    {"group_id": "form-01", "group_id_predicted": "form-01", "page_number": 5, "page_number_predicted": 4, "class_label": "form", "class_label_predicted": "form"},
]

r = evaluate_packet(wrong_order_pages, strict_clustering=True)
print(f"Packet Score:    {r['final_score']:.4f}")
print(f"Clustering:      {r['clustering_score']:.4f}  (perfect — groups are correct)")
print(f"Ordering:        {r['avg_ordering_score']:.4f}  (negative — pages scrambled)")

### Edge Case: Reverse Order

Pages in completely reversed sequence. Proposed metrics distinguish this from partial scrambling.

In [ ]:
reverse_pages = [
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 1, "page_number_predicted": 3, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 2, "page_number_predicted": 2, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 3, "page_number_predicted": 1, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "form-01", "group_id_predicted": "form-01", "page_number": 4, "page_number_predicted": 5, "class_label": "form", "class_label_predicted": "form"},
    {"group_id": "form-01", "group_id_predicted": "form-01", "page_number": 5, "page_number_predicted": 4, "class_label": "form", "class_label_predicted": "form"},
]

r = evaluate_packet(reverse_pages, strict_clustering=True)
print(f"Packet Score:    {r['final_score']:.4f}")
print(f"Ordering:        {r['avg_ordering_score']:.4f}  (Kendall's Tau = -1.0 for full reversal)")

### Comparing Over-segmentation vs Under-segmentation

Classical metrics score both identically (0%). Proposed metrics distinguish severity.

In [ ]:
# Over-segmentation: one document split into two
split_pages = [
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 1, "page_number_predicted": 1, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "inv-01", "group_id_predicted": "inv-02", "page_number": 2, "page_number_predicted": 2, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "inv-01", "group_id_predicted": "inv-01", "page_number": 3, "page_number_predicted": 3, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "form-01", "group_id_predicted": "form-01", "page_number": 4, "page_number_predicted": 4, "class_label": "form", "class_label_predicted": "form"},
    {"group_id": "form-01", "group_id_predicted": "form-02", "page_number": 5, "page_number_predicted": 5, "class_label": "form", "class_label_predicted": "form"},
]

# Under-segmentation: two documents merged into one
merged_pages = [
    {"group_id": "inv-01", "group_id_predicted": "mixed-01", "page_number": 1, "page_number_predicted": 1, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "inv-01", "group_id_predicted": "mixed-01", "page_number": 2, "page_number_predicted": 2, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "inv-01", "group_id_predicted": "mixed-01", "page_number": 3, "page_number_predicted": 3, "class_label": "invoice", "class_label_predicted": "invoice"},
    {"group_id": "form-01", "group_id_predicted": "mixed-01", "page_number": 4, "page_number_predicted": 4, "class_label": "form", "class_label_predicted": "form"},
    {"group_id": "form-01", "group_id_predicted": "mixed-01", "page_number": 5, "page_number_predicted": 5, "class_label": "form", "class_label_predicted": "form"},
]

r_split = evaluate_packet(split_pages, strict_clustering=True)
r_merged = evaluate_packet(merged_pages, strict_clustering=True)

print("Over-segmentation (split):")
print(f"  Packet Score:  {r_split['final_score']:.4f}")
print(f"  Clustering:    {r_split['clustering_score']:.4f}")
print()
print("Under-segmentation (merged):")
print(f"  Packet Score:  {r_merged['final_score']:.4f}")
print(f"  Clustering:    {r_merged['clustering_score']:.4f}")
print()
print("Under-segmentation is penalized more heavily (0.20 vs 0.69 clustering)")
print("because merging permanently loses document boundaries.")

### Tuning Weights

In [ ]:
# Same data, different weight configurations
data = wrong_order_pages

r_default = evaluate_packet(data, alpha=0.5, beta=0.5, strict_clustering=True)
r_cluster = evaluate_packet(data, alpha=0.7, beta=0.3, strict_clustering=True)
r_order = evaluate_packet(data, alpha=0.3, beta=0.7, strict_clustering=True)

print(f"Default (α=0.5, β=0.5):           {r_default['final_score']:.4f}")
print(f"Clustering-heavy (α=0.7, β=0.3):  {r_cluster['final_score']:.4f}")
print(f"Ordering-heavy (α=0.3, β=0.7):    {r_order['final_score']:.4f}")

### Strict vs Non-Strict Clustering

In [ ]:
r_strict = evaluate_packet(misclass_pages, strict_clustering=True)
r_nonstrict = evaluate_packet(misclass_pages, strict_clustering=False)

print("Misclassification scenario:")
print(f"  Strict mode:     Packet={r_strict['final_score']:.4f}, Clustering={r_strict['clustering_score']:.4f}")
print(f"  Non-strict mode: Packet={r_nonstrict['final_score']:.4f}, Clustering={r_nonstrict['clustering_score']:.4f}")
print()
print("Non-strict mode ignores class labels — only evaluates grouping structure.")